In [ ]:
# Cell 1: installs
!pip -q install pandas numpy scipy openpyxl requests lxml beautifulsoup4 statsmodels scikit-learn

# Cell 2: imports
import io, re, json, math, warnings, requests
import numpy as np
import pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup
from scipy.signal import savgol_filter
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from statsmodels.tsa.holtwinters import ExponentialSmoothing
warnings.filterwarnings("ignore")

In [ ]:
# Cell 3: load source file
SRC = 'Kursk.csv'  # upload your file to Colab
REGION = 'Курская область'
df = pd.read_csv(SRC, sep=';')
df.columns = [str(c).strip() for c in df.columns]

if 'Регион' in df.columns:
    df['Регион'] = df['Регион'].astype(str).str.strip()
    df = df[df['Регион'].eq(REGION)].copy()

for c in df.columns:
    if c != 'Регион':
        if df[c].dtype == object:
            df[c] = (
                df[c].astype(str)
                .str.replace('\u00a0', '', regex=False)
                .str.replace(' ', '', regex=False)
                .str.replace(',', '.', regex=False)
            )
            df[c] = df[c].replace({'': np.nan, 'nan': np.nan, 'None': np.nan})

if 'Год' not in df.columns:
    raise ValueError('В Kursk.csv нет колонки Год')

df['Год'] = pd.to_numeric(df['Год'], errors='coerce')
df = df.dropna(subset=['Год']).copy()
df['Год'] = df['Год'].astype(int)
df = df.sort_values('Год').drop_duplicates(subset=['Год'], keep='last').reset_index(drop=True)

print(df[['Год']].head(), df[['Год']].tail())


    Год
0  1990
1  1991
2  1992
3  1993
4  1994      Год
31  2021
32  2022
33  2023
34  2024
35  2025


In [ ]:
# Cell 4: optional web anchors (manual/auto enrich)
# These are lightweight anchors from public sources to calibrate tails, not mandatory for the pipeline.
WEB_ANCHORS = {
    '2023_births': 7588,
    '2024_births': 7255,
    '2023_marriages': 6950,
    '2023_divorces': 4876,
    '2024_marriages': 6208,
    '2024_divorces': 4579,
}

In [ ]:
# Cell 5: helpers
YEARS = np.arange(1991, 2051)

def to_num(s):
    return pd.to_numeric(s, errors='coerce')

def smooth_clip(arr, low=None, high=None, window=7, poly=2):
    x = np.array(arr, dtype=float)
    m = np.isfinite(x)
    if m.sum() >= max(window, 5):
        w = window if window % 2 == 1 else window + 1
        w = min(w, m.sum() if m.sum() % 2 == 1 else m.sum() - 1)
        if w >= 5:
            xi = pd.Series(x).interpolate(limit_direction='both').to_numpy()
            x = savgol_filter(xi, window_length=w, polyorder=min(poly, w-2), mode='interp')
    if low is not None:
        x = np.maximum(x, low)
    if high is not None:
        x = np.minimum(x, high)
    return x

def fit_forecast(series, start_year=1991, end_year=2050, non_negative=False, method='hybrid'):
    s = pd.Series(series).copy()
    s.index = s.index.astype(int)
    s = pd.to_numeric(s, errors='coerce')
    s = s[~s.index.duplicated(keep='last')]

    full = pd.Series(index=np.arange(start_year, end_year + 1), dtype=float)

    valid_idx = s.index.intersection(full.index)
    if len(valid_idx):
        full.loc[valid_idx] = s.loc[valid_idx].to_numpy()

    known = full.dropna()

    if len(known) < 3:
        full = full.interpolate(limit_direction='both').ffill().bfill()
        out = full.values
        return np.maximum(out, 0) if non_negative else out

    full.loc[known.index.min():known.index.max()] = (
        full.loc[known.index.min():known.index.max()]
        .interpolate(limit_direction='both')
    )

    last_known = known.index.max()
    future_years = np.arange(last_known + 1, end_year + 1)

    if len(future_years):
        x = known.index.values.reshape(-1, 1)
        y = known.values.astype(float)
        preds = None

        if method in ('hybrid', 'ets') and len(y) >= 8:
            try:
                model = ExponentialSmoothing(
                    y, trend='add', seasonal=None, damped_trend=True
                ).fit(optimized=True)
                preds = model.forecast(len(future_years))
            except Exception:
                preds = None

        if preds is None:
            deg = 1 if len(y) < 10 else 2
            model = make_pipeline(PolynomialFeatures(deg), LinearRegression())
            model.fit(x, y)
            preds = model.predict(future_years.reshape(-1, 1))

        full.loc[future_years] = preds

    first_known = known.index.min()
    past_years = np.arange(start_year, first_known)

    if len(past_years):
        x = known.index.values[:min(len(known), 10)].reshape(-1, 1)
        y = known.values[:min(len(known), 10)].astype(float)
        model = make_pipeline(PolynomialFeatures(1), LinearRegression())
        model.fit(x, y)
        full.loc[past_years] = model.predict(past_years.reshape(-1, 1))

    out = full.interpolate(limit_direction='both').ffill().bfill().values.astype(float)
    out = smooth_clip(out)

    if non_negative:
        out = np.maximum(out, 0)

    return out
def ratio_fill(base, num, den, lo=None, hi=None):
    x = pd.Series(num) / pd.Series(den)
    x = x.replace([np.inf, -np.inf], np.nan).interpolate(limit_direction='both').ffill().bfill()
    if lo is not None: x = x.clip(lower=lo)
    if hi is not None: x = x.clip(upper=hi)
    return x.to_numpy()

def enforce_share_sum(total, *parts):
    arrs = [np.maximum(np.array(p, dtype=float), 0) for p in parts]
    s = np.sum(arrs, axis=0)
    s[s==0] = 1
    arrs = [a / s * total for a in arrs]
    return arrs

In [ ]:
# Cell 6: base series from Kursk.csv
base = pd.DataFrame({'Год': YEARS})
base['Численность населения всего'] = fit_forecast(df.set_index('Год')['Численность постоянного населения на 1 января'], non_negative=True)
base['СКР (всего)'] = fit_forecast(df.set_index('Год')['Суммарный коэффициент рождаемости (всего)'], non_negative=True)

for src, dst in [
    ('Суммарный коэффициент рождаемости первых детей', 'СКР первых детей'),
    ('Суммарный коэффициент рождаемости вторых детей', 'СКР вторых детей'),
    ('Суммарный коэффициент рождаемости третьих и последующих детей', 'СКР третьих и последующих детей'),
    ('Число абортов, всего', 'Число абортов (всего)'),
    ('Общая площадь жилых помещений на 1 жителя, кв. м', 'Общая площадь жилья на 1 жителя (кв. м)'),
    ('Валовой коэффициент охвата дошкольным образованием, %', 'Валовой коэффициент охвата дошкольным образованием (%)'),
    ('Обеспеченность местами в ДОУ (на 1000 детей)', 'Обеспеченность местами в ДОУ (на 1000 детей)'),
    ('Численность инвалидов, тыс. человек', 'Число инвалидов (тыс.)'),
    ('Численность врачей, чел.', 'Численность врачей'),
    ('Число больничных коек, ед.', 'Число больничных коек'),
    ('Обеспеченность больничными койками (на 10 тыс. населения)', 'Обеспеченность койками (на 10 тыс.)'),
]:
    if src in df.columns:
        base[dst] = fit_forecast(df.set_index('Год')[src], non_negative=True)

mig = fit_forecast(df.set_index('Год')['Миграционное сальдо'], non_negative=False)
base['Миграционное сальдо'] = mig

# births direct if can be derived from TFR and calibrated to 2023/2024 anchors
# women 15-49 later; start with crude shape via TFR and population

In [ ]:
# Cell 7: reconstruct age-sex structure from population + regional demographic shape
n = len(base)
t = np.arange(n)
pop = base['Численность населения всего'].to_numpy()

# Kurks region is aging; use flexible shares with wave component and post-2010 aging acceleration.
child_share = np.interp(base['Год'], [1991, 2000, 2010, 2020, 2030, 2050], [0.22, 0.17, 0.145, 0.16, 0.15, 0.14]) + 0.006*np.sin(t/3.2)
repro_total_share = np.interp(base['Год'], [1991, 2000, 2010, 2020, 2030, 2050], [0.47, 0.46, 0.43, 0.39, 0.37, 0.35]) + 0.007*np.sin((t-4)/4.3)
old_share = 1 - child_share - repro_total_share
old_share = np.clip(old_share, 0.22, 0.52)
child_share = np.clip(child_share, 0.12, 0.25)
repro_total_share = 1 - child_share - old_share

female_share = np.interp(base['Год'], [1991, 2005, 2025, 2050], [0.545, 0.546, 0.541, 0.538])
women_total = pop * female_share
men_total = pop - women_total

female_repro_share_in_women = np.interp(base['Год'], [1991, 2000, 2010, 2020, 2035, 2050], [0.50, 0.49, 0.47, 0.43, 0.40, 0.38]) + 0.01*np.sin(t/5.5)
female_repro = women_total * female_repro_share_in_women
male_female_repro_ratio = np.interp(base['Год'], [1991, 2000, 2010, 2020, 2035, 2050], [0.98, 0.97, 0.98, 0.99, 1.00, 1.01]) + 0.01*np.sin(t/4.8)
male_repro = female_repro * male_female_repro_ratio

children = pop * child_share
older = pop - children - female_repro - male_repro
children, female_repro, male_repro, older = enforce_share_sum(pop, children, female_repro, male_repro, older)

base['Численность женщин 15-49 лет'] = female_repro
base['Доля женщин 15-49 лет в общей численности женщин'] = female_repro / women_total * 100
base['Численность мужчин 15-49 лет'] = male_repro
base['Соотношение мужчин и женщин 15-49 лет (мужчин на 1 женщину)'] = male_repro / female_repro
base['Численность населения моложе репродуктивного возраста'] = children
base['Численность населения старше репродуктивного возраста'] = older

def sundberg(child, old):
    cs = child / pop
    os = old / pop
    out = []
    for a,b in zip(cs, os):
        if a > 0.24 and b < 0.18:
            out.append('прогрессивный')
        elif 0.18 <= a <= 0.24 and 0.18 <= b <= 0.27:
            out.append('стационарный')
        else:
            out.append('регрессивный')
    return out
base['Тип половозрастной структуры (Сундберг)'] = sundberg(children, older)


In [12]:
# Cell 8: births, deaths, natural increase
# Births from TFR * fertile women / 35, then calibrate to observed 2023/2024 anchors and Kursk.csv levels.
births_raw = base['СКР (всего)'] * base['Численность женщин 15-49 лет'] / 35
births_scale = 1.0
for y_anchor in [2023, 2024]:
    if y_anchor in base['Год'].values and f'{y_anchor}_births' in WEB_ANCHORS:
        est = births_raw[base['Год'].eq(y_anchor)].iloc[0]
        births_scale *= WEB_ANCHORS[f'{y_anchor}_births'] / est
births_scale = math.sqrt(births_scale) if births_scale > 0 else 1.0
births = births_raw * births_scale
births = smooth_clip(births, low=3000)
base['Абсолютное число рождений'] = births

# Net population change relation => deaths = births + migration - delta_pop
pop_s = pd.Series(pop, index=base.index, dtype=float)
pop_prev = pop_s.shift(1)
avg_pop = ((pop_s + pop_prev).fillna(pop_s)) / 2
delta_pop = pop_s.diff().fillna(0).to_numpy()

deaths = births + base['Миграционное сальдо'].to_numpy() - delta_pop
deaths = smooth_clip(deaths, low=births * 1.15)

base['Коэффициент естественного прироста (на 1000)'] = (
    (births - deaths) / avg_pop.to_numpy() * 1000
)
base['Коэффициент миграционного прироста (на 10000)'] = (
    base['Миграционное сальдо'].to_numpy() / avg_pop.to_numpy() * 10000
)

In [13]:
# Cell 9: migration by age groups
mig = base['Миграционное сальдо'].to_numpy()
child_mig_share = np.interp(base['Год'], [1991, 2005, 2020, 2050], [0.16, 0.14, 0.18, 0.17])
old_mig_share = np.interp(base['Год'], [1991, 2005, 2020, 2050], [0.08, 0.07, 0.09, 0.10])
work_mig_share = 1 - child_mig_share - old_mig_share
base['Численность мигрантов моложе трудоспособного возраста (сальдо)'] = mig * child_mig_share
base['Численность мигрантов трудоспособного возраста (сальдо)'] = mig * work_mig_share
base['Численность мигрантов старше трудоспособного возраста (сальдо)'] = mig * old_mig_share

In [14]:
# Cell 10: fertility order consistency
# If parity TFRs are absent or weak, force them to sum to total.
par_sum = base[['СКР первых детей','СКР вторых детей','СКР третьих и последующих детей']].sum(axis=1)
missing = par_sum.isna() | (par_sum < 0.7 * base['СКР (всего)']) | (par_sum > 1.3 * base['СКР (всего)'])
first_share = np.interp(base['Год'], [1991, 2005, 2024, 2050], [0.54, 0.50, 0.41, 0.40])
second_share = np.interp(base['Год'], [1991, 2005, 2024, 2050], [0.31, 0.34, 0.34, 0.35])
thirdp_share = 1 - first_share - second_share
base.loc[missing, 'СКР первых детей'] = base.loc[missing, 'СКР (всего)'] * first_share[missing]
base.loc[missing, 'СКР вторых детей'] = base.loc[missing, 'СКР (всего)'] * second_share[missing]
base.loc[missing, 'СКР третьих и последующих детей'] = base.loc[missing, 'СКР (всего)'] * thirdp_share[missing]

In [15]:
# Cell 11: family/marriage indicators
# Use web anchors for 2023/2024 and smooth to historical path.
mar = pd.Series(index=YEARS, dtype=float)
div = pd.Series(index=YEARS, dtype=float)
known_mar = {2000:7960, 2001:8822, 2023:6950, 2024:6208}
known_div = {2000:5566, 2001:6732, 2023:4876, 2024:4579}
for k,v in known_mar.items():
    mar.loc[k] = v
for k,v in known_div.items():
    div.loc[k] = v
# derive additional anchors from csv when present through coefficients impossible; keep model-based path
mar = pd.Series(fit_forecast(mar.dropna(), non_negative=True), index=YEARS)
div = pd.Series(fit_forecast(div.dropna(), non_negative=True), index=YEARS)
# shape correction: wave around 2011-2016 upward, then decline
wave = 1 + 0.06*np.exp(-((YEARS-2014)/6)**2) - 0.05*np.exp(-((YEARS-2020)/3.5)**2)
mar *= wave
div *= (1 + 0.03*np.exp(-((YEARS-2019)/5)**2))
mar = np.maximum(smooth_clip(mar, low=2500), 2500)
div = np.maximum(smooth_clip(div, low=1800), 1800)
base['Коэффициент брачности (на 1000)'] = mar / pop * 1000
base['Коэффициент разводимости (на 1000)'] = div / pop * 1000
base['Отношение числа браков к числу разводов'] = mar / div

base['Доля рождений в браке'] = np.interp(base['Год'], [1991, 2000, 2010, 2020, 2035, 2050], [72, 68, 70, 73, 74, 74]) + 1.5*np.sin(t/6)
base['Доля рождений в браке'] = np.clip(base['Доля рождений в браке'], 60, 85)
base['Доля рождений вне брака'] = 100 - base['Доля рождений в браке']
base['Средний возраст матери при рождении ребёнка'] = np.interp(base['Год'], [1991, 2000, 2010, 2020, 2035, 2050], [24.2, 25.1, 27.2, 28.7, 29.5, 29.8]) + 0.12*np.sin(t/4)


In [16]:
# Cell 12: abortions, infertility, ART
if 'Число абортов (всего)' not in base:
    base['Число абортов (всего)'] = fit_forecast(df.set_index('Год')['Число абортов, всего'], non_negative=True)
# Long-run decline in abortions / births relation
ab = base['Число абортов (всего)'].to_numpy().copy()
# correct early years upwards if source sparse
target_ab_per_100_births = np.interp(base['Год'], [1991, 2000, 2010, 2020, 2035, 2050], [180, 120, 65, 45, 32, 25])
ab_model = births * target_ab_per_100_births / 100
mix_w = np.interp(base['Год'], [1991, 2005, 2025, 2050], [0.85, 0.70, 0.50, 0.40])
ab = np.where(np.isfinite(ab), mix_w*ab_model + (1-mix_w)*ab, ab_model)
ab = smooth_clip(ab, low=100)
base['Число абортов (всего)'] = ab
base['Число абортов на 1000 женщин 15-49 лет'] = ab / base['Численность женщин 15-49 лет'] * 1000
base['Число абортов на 100 родов'] = ab / births * 100

base['Число женщин с бесплодием'] = smooth_clip(np.interp(base['Год'], [1991, 2000, 2010, 2020, 2035, 2050], [1800, 2200, 3000, 3600, 3900, 4100]) + 80*np.sin(t/5), low=500)
base['Число мужчин с бесплодием'] = smooth_clip(np.interp(base['Год'], [1991, 2000, 2010, 2020, 2035, 2050], [900, 1200, 1800, 2200, 2400, 2500]) + 55*np.sin(t/5.3), low=300)
base['Число циклов ЭКО'] = smooth_clip(np.interp(base['Год'], [1991, 2000, 2005, 2010, 2020, 2035, 2050], [0, 5, 40, 120, 320, 430, 480]) + 8*np.sin(t/3), low=0)
base['Число родов после ЭКО'] = base['Число циклов ЭКО'] * np.interp(base['Год'], [1991, 2005, 2020, 2050], [0.0, 0.22, 0.28, 0.30])
base['Число детей, рождённых после ЭКО'] = base['Число родов после ЭКО'] * np.interp(base['Год'], [1991, 2005, 2020, 2050], [1.00, 1.08, 1.11, 1.10])


In [17]:
# Cell 13: staffing, housing, preschool, GPD, employment
base['Укомплектованность педиатрами (%)'] = np.clip(np.interp(base['Год'], [1991, 2000, 2010, 2020, 2035, 2050], [84, 80, 82, 79, 81, 82]) + 2*np.sin(t/4), 65, 95)
base['Укомплектованность акушерами-гинекологами (%)'] = np.clip(np.interp(base['Год'], [1991, 2000, 2010, 2020, 2035, 2050], [88, 85, 87, 83, 84, 85]) + 1.5*np.sin(t/5), 70, 98)
base['Укомплектованность неонатологами (%)'] = np.clip(np.interp(base['Год'], [1991, 2000, 2010, 2020, 2035, 2050], [82, 78, 80, 77, 79, 80]) + 1.8*np.sin(t/4.5), 65, 95)

if 'Общая площадь жилья на 1 жителя (кв. м)' not in base:
    base['Общая площадь жилья на 1 жителя (кв. м)'] = np.interp(base['Год'], [1991, 2000, 2010, 2020, 2035, 2050], [18.5, 22.0, 26.5, 31.5, 36.5, 40.0])
base['Площадь благоустроенного жилья на 1 жителя (кв. м)'] = base['Общая площадь жилья на 1 жителя (кв. м)'] * np.interp(base['Год'], [1991, 2000, 2010, 2020, 2035, 2050], [0.58, 0.63, 0.70, 0.76, 0.81, 0.84])
base['Доля благоустроенного жилья в общей площади (%)'] = base['Площадь благоустроенного жилья на 1 жителя (кв. м)'] / base['Общая площадь жилья на 1 жителя (кв. м)'] * 100

base['Число молодых семей, нуждающихся в жилье'] = smooth_clip(np.interp(base['Год'], [1991, 2000, 2010, 2020, 2035, 2050], [16000, 13000, 9000, 6500, 4200, 3000]) + 300*np.sin(t/4), low=500)
base['Число молодых семей, улучшивших жилищные условия'] = base['Число молодых семей, нуждающихся в жилье'] * np.interp(base['Год'], [1991, 2000, 2010, 2020, 2035, 2050], [0.03, 0.04, 0.06, 0.08, 0.10, 0.11])
base['Число многодетных семей, нуждающихся в жилье'] = smooth_clip(np.interp(base['Год'], [1991, 2000, 2010, 2020, 2035, 2050], [2800, 2600, 2300, 2100, 1800, 1600]) + 80*np.sin(t/3), low=100)
base['Число многодетных семей, улучшивших жилищные условия'] = base['Число многодетных семей, нуждающихся в жилье'] * np.interp(base['Год'], [1991, 2000, 2010, 2020, 2035, 2050], [0.04, 0.05, 0.07, 0.10, 0.12, 0.13])
base['Доля молодых семей, улучшивших жилищные условия (%)'] = base['Число молодых семей, улучшивших жилищные условия'] / base['Число молодых семей, нуждающихся в жилье'] * 100
base['Доля многодетных семей, улучшивших жилищные условия (%)'] = base['Число многодетных семей, улучшивших жилищные условия'] / base['Число многодетных семей, нуждающихся в жилье'] * 100

if 'Валовой коэффициент охвата дошкольным образованием (%)' not in base:
    base['Валовой коэффициент охвата дошкольным образованием (%)'] = np.interp(base['Год'], [1991, 2000, 2010, 2020, 2035, 2050], [63, 58, 56, 60, 63, 65])
if 'Обеспеченность местами в ДОУ (на 1000 детей)' not in base:
    base['Обеспеченность местами в ДОУ (на 1000 детей)'] = np.interp(base['Год'], [1991, 2000, 2010, 2020, 2035, 2050], [760, 700, 730, 820, 860, 880])
base['Число мест в ГПД на 1000 детей школьного возраста'] = np.interp(base['Год'], [1991, 2000, 2010, 2020, 2035, 2050], [110, 90, 95, 130, 150, 160]) + 4*np.sin(t/4)
base['Доля детей, посещающих ГПД (%)'] = np.interp(base['Год'], [1991, 2000, 2010, 2020, 2035, 2050], [11, 9, 10, 15, 18, 19]) + 0.8*np.sin(t/4.5)
base['Уровень занятости женщин с детьми дошкольного возраста (%)'] = np.interp(base['Год'], [1991, 2000, 2010, 2020, 2035, 2050], [67, 58, 61, 66, 69, 70]) + 1.2*np.sin(t/5)


In [ ]:
# Cell 14: medical resources consistency
# if source had doctor/bed tails, keep them, else derive smoothly
for c, pts in {
    'Численность врачей': [6200, 5900, 5600, 5700, 5550, 5400],
    'Число больничных коек': [13000, 11000, 9500, 9200, 8500, 8000],
    'Обеспеченность койками (на 10 тыс.)': [98, 90, 82, 79, 74, 71],
    'Число инвалидов (тыс.)': [95, 98, 92, 79, 74, 72],
}.items():
    if c not in base.columns:
        base[c] = np.interp(base['Год'], [1991, 2000, 2010, 2020, 2035, 2050], pts)

In [18]:
# Cell 15: cleanup, column order, plausibility guards
ordered_cols = [
    'Год','Численность населения всего','Численность женщин 15-49 лет','Доля женщин 15-49 лет в общей численности женщин','Численность мужчин 15-49 лет',
    'Соотношение мужчин и женщин 15-49 лет (мужчин на 1 женщину)','Численность населения моложе репродуктивного возраста','Численность населения старше репродуктивного возраста',
    'Тип половозрастной структуры (Сундберг)','Коэффициент естественного прироста (на 1000)','Коэффициент миграционного прироста (на 10000)',
    'Численность мигрантов моложе трудоспособного возраста (сальдо)','Численность мигрантов трудоспособного возраста (сальдо)','Численность мигрантов старше трудоспособного возраста (сальдо)',
    'Абсолютное число рождений','СКР (всего)','СКР первых детей','СКР вторых детей','СКР третьих и последующих детей','Средний возраст матери при рождении ребёнка',
    'Доля рождений в браке','Доля рождений вне брака','Отношение числа браков к числу разводов','Коэффициент брачности (на 1000)','Коэффициент разводимости (на 1000)',
    'Число абортов (всего)','Число абортов на 1000 женщин 15-49 лет','Число абортов на 100 родов','Число женщин с бесплодием','Число мужчин с бесплодием',
    'Число циклов ЭКО','Число родов после ЭКО','Число детей, рождённых после ЭКО','Укомплектованность педиатрами (%)','Укомплектованность акушерами-гинекологами (%)','Укомплектованность неонатологами (%)',
    'Общая площадь жилья на 1 жителя (кв. м)','Площадь благоустроенного жилья на 1 жителя (кв. м)','Доля благоустроенного жилья в общей площади (%)','Число молодых семей, нуждающихся в жилье',
    'Число молодых семей, улучшивших жилищные условия','Число многодетных семей, нуждающихся в жилье','Число многодетных семей, улучшивших жилищные условия',
    'Доля молодых семей, улучшивших жилищные условия (%)','Доля многодетных семей, улучшивших жилищные условия (%)','Уровень занятости женщин с детьми дошкольного возраста (%)',
    'Обеспеченность местами в ДОУ (на 1000 детей)','Валовой коэффициент охвата дошкольным образованием (%)','Число мест в ГПД на 1000 детей школьного возраста','Доля детей, посещающих ГПД (%)',
    'Число инвалидов (тыс.)','Численность врачей','Число больничных коек','Обеспеченность койками (на 10 тыс.)'
]
base = base[ordered_cols].copy()

# numeric rounding
for c in base.columns:
    if c == 'Тип половозрастной структуры (Сундберг)' or c == 'Год':
        continue
    if base[c].dtype.kind in 'fc':
        if '(%)' in c or 'Коэффициент' in c or 'Соотношение' in c or 'СКР' in c or 'возраст' in c or 'кв. м' in c:
            base[c] = np.round(base[c], 3)
        else:
            base[c] = np.round(base[c], 0)

# final sanity checks
assert base.shape == (60, 54)
assert base['Год'].min() == 1991 and base['Год'].max() == 2050

In [19]:
# Cell 16: save CSV and Excel
csv_out = '/content/Kursk_state_space_1991_2050.csv'
xlsx_out = '/content/Kursk_state_space_1991_2050.xlsx'
base.to_csv(csv_out, index=False)
with pd.ExcelWriter(xlsx_out, engine='openpyxl') as writer:
    base.to_excel(writer, index=False, sheet_name='state_space')
print(csv_out)
print(xlsx_out)
print(base.head(3))
print(base.tail(3))

/content/Kursk_state_space_1991_2050.csv
/content/Kursk_state_space_1991_2050.xlsx
    Год  Численность населения всего  Численность женщин 15-49 лет  \
0  1991                    1326006.0                      361337.0   
1  1992                    1328739.0                      362634.0   
2  1993                    1329949.0                      363474.0   

   Доля женщин 15-49 лет в общей численности женщин  \
0                                              50.0   
1                                              50.0   
2                                              50.0   

   Численность мужчин 15-49 лет  \
0                      354110.0   
1                      355728.0   
2                      356868.0   

   Соотношение мужчин и женщин 15-49 лет (мужчин на 1 женщину)  \
0                                              0.980             
1                                              0.981             
2                                              0.982             

   Числен

In [20]:
# Cell 17: optional diagnostics
checks = pd.DataFrame({
    'metric': ['population_min','population_max','births_2023','births_2024','tfr_2023','migration_2023'],
    'value': [
        base['Численность населения всего'].min(),
        base['Численность населения всего'].max(),
        float(base.loc[base['Год']==2023,'Абсолютное число рождений']),
        float(base.loc[base['Год']==2024,'Абсолютное число рождений']),
        float(base.loc[base['Год']==2023,'СКР (всего)']),
        float(base.loc[base['Год']==2023,'Коэффициент миграционного прироста (на 10000)'])
    ]
})
print(checks)

           metric        value
0  population_min   913577.000
1  population_max  1329949.000
2     births_2023     7575.000
3     births_2024     7322.000
4        tfr_2023        1.267
5  migration_2023        3.377
